In [1]:
import json
from pathlib import Path
from datetime import datetime

JSON_DIR = Path("data/processed/extracted_json")
VALIDATED_DIR = Path("data/processed/validated")
VALIDATED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
## Validation Rules

VALID_VAT_RATES = [0, 7, 19]  # German VAT rates (Regelsatz 19%, ermaessigt 7%, or 0%)

def validate_invoice(data, filename):
    errors = []
    warnings = []

    # 1. Required fields present
    required_fields = ["vendor", "invoice_number", "invoice_date",
                        "net_amount", "vat_rate", "vat_amount",
                        "total_amount", "currency"]
    for field in required_fields:
        if data.get(field) is None:
            errors.append(f"Missing field: {field}")

    # Stop early if critical numeric fields are missing
    if errors:
        return errors, warnings

    # 2. VAT rate is a valid German rate
    if data["vat_rate"] not in VALID_VAT_RATES:
        warnings.append(f"Unusual VAT rate: {data['vat_rate']}% (expected 0, 7, or 19)")

    # 3. VAT math checks out
    expected_vat = round(data["net_amount"] * data["vat_rate"] / 100, 2)
    if abs(expected_vat - data["vat_amount"]) > 0.02:  # small rounding tolerance
        errors.append(
            f"VAT amount mismatch: expected {expected_vat}, got {data['vat_amount']}"
        )

    # 4. Total math checks out
    expected_total = round(data["net_amount"] + data["vat_amount"], 2)
    if abs(expected_total - data["total_amount"]) > 0.02:
        errors.append(
            f"Total mismatch: expected {expected_total}, got {data['total_amount']}"
        )

    # 5. Date format is valid and not in the future
    try:
        parsed_date = datetime.strptime(data["invoice_date"], "%Y-%m-%d")
        if parsed_date > datetime.now():
            warnings.append(f"Invoice date is in the future: {data['invoice_date']}")
    except (ValueError, TypeError):
        errors.append(f"Invalid date format: {data.get('invoice_date')}")

    # 6. Amounts are positive
    for amount_field in ["net_amount", "vat_amount", "total_amount"]:
        if data[amount_field] < 0:
            errors.append(f"Negative value in {amount_field}: {data[amount_field]}")

    # 7. Currency looks like a valid code
    if data["currency"] not in ["EUR", "USD", "GBP", "CHF"]:
        warnings.append(f"Unusual currency code: {data['currency']}")

    return errors, warnings

In [3]:
## Ran validation on all extracted JSON files

def validate_all_invoices():
    summary = []
    json_files = list(JSON_DIR.glob("*.json"))

    if not json_files:
        print("No JSON files found in", JSON_DIR)
        return summary

    for json_file in json_files:
        data = json.loads(json_file.read_text(encoding="utf-8"))
        errors, warnings = validate_invoice(data, json_file.name)

        status = "FAILED" if errors else ("WARNING" if warnings else "PASSED")

        result = {
            "filename": json_file.name,
            "status": status,
            "errors": errors,
            "warnings": warnings,
            "data": data
        }
        summary.append(result)

        # Save validated (or flagged) record
        out_path = VALIDATED_DIR / json_file.name
        out_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")

        print(f"[{status}] {json_file.name}")
        if errors:
            for e in errors:
                print(f"   ERROR: {e}")
        if warnings:
            for w in warnings:
                print(f"   WARNING: {w}")
        print()

    return summary


validation_results = validate_all_invoices()

[PASSED] invoice_001.json

[PASSED] invoice_003.json

[PASSED] invoice_002.json



In [4]:
## Summary Report

def print_summary(results):
    total = len(results)
    passed = sum(1 for r in results if r["status"] == "PASSED")
    warned = sum(1 for r in results if r["status"] == "WARNING")
    failed = sum(1 for r in results if r["status"] == "FAILED")

    print("=" * 40)
    print("VALIDATION SUMMARY")
    print("=" * 40)
    print(f"Total invoices:  {total}")
    print(f"Passed:          {passed}")
    print(f"Warnings:        {warned}")
    print(f"Failed:          {failed}")
    print("=" * 40)


print_summary(validation_results)

VALIDATION SUMMARY
Total invoices:  3
Passed:          3
Warnings:        0
Failed:          0


In [ ]:
## Step 3: Data Validation (`03_validate.ipynb`)

### What this notebook does
This is the quality-control stage of the pipeline. It takes the structured JSON produced in Step 2 and checks whether the extracted data is actually correct and internally consistent — before anything gets stored as "trusted" data.

### Why this step exists
LLM extraction is powerful but not infallible — it can misread a number, pick the wrong field, or occasionally hallucinate a value. Rather than trusting the output blindly, this step applies deterministic business-rule checks on top of the AI output, catching errors the model itself can't self-verify.

### Process
1. **Define validation rules** based on real German invoicing logic:
   - Required fields are present (vendor, invoice number, date, amounts, etc.)
   - VAT rate matches a valid German rate (0%, 7%, or 19%)
   - VAT amount math checks out (`net_amount × vat_rate/100 ≈ vat_amount`)
   - Total math checks out (`net_amount + vat_amount ≈ total_amount`)
   - Invoice date is valid and not set in the future
   - Amounts are non-negative
   - Currency code looks valid
2. **Run validation on every extracted invoice** and classify each as:
   - **PASSED** — no issues
   - **WARNING** — minor irregularities (e.g. unusual VAT rate) that don't block processing but are worth a human glance
   - **FAILED** — hard errors (e.g. VAT math doesn't add up, missing required field) that should not be trusted without review
3. **Save validated records** — each invoice's result (data + errors + warnings + status) is saved to `data/processed/validated/`, preserving a full audit trail of what was checked and why.
4. **Print a summary report** — a quick pass/warning/fail count across all processed invoices.

### Why this matters for a real business use case
This is the difference between a toy demo and something a business could actually trust: automated extraction without validation risks silently feeding wrong numbers into accounting. Flagging math mismatches or invalid VAT rates before storage means a human only needs to review the flagged exceptions, not every single invoice — this is where the real efficiency gain of the pipeline comes from.

### Output
- `data/processed/validated/` — one JSON file per invoice containing the original extracted data plus its validation status, errors, and warnings
